# CSBP711 A1 — run the full experiment on Colab

Runtime → Change runtime type → **GPU (T4)**. Run the cells top to bottom (≈ 1 h per seed). Every cell uses absolute paths, so re-running one is safe.

## 0. Get the code — pick ONE of the two cells
**A.** Upload `cxr-algo-compare.zip` (the repository zip).  **B.** Clone the group's GitHub repo once it exists.

In [ ]:
# A) upload the zip
import os
from google.colab import files
os.chdir('/content')
!rm -rf /content/repo /content/cxr-algo-compare
up = files.upload()                      # choose cxr-algo-compare.zip
!unzip -q /content/cxr-algo-compare.zip -d /content && mv /content/cxr-algo-compare /content/repo
os.chdir('/content/repo')
!pip install -q -r requirements.txt
!nvidia-smi -L && ls

In [ ]:
# B) clone from GitHub (skip if you used A)
import os
REPO_URL = 'https://github.com/mahdhindi/pneumonia-cxr-algorithm-comparison.git'
os.chdir('/content')
!rm -rf /content/repo
BRANCH = 'a2-paper-critique'                 # <- Assignment 2 branch; use 'main' after it is merged
!git clone -b $BRANCH $REPO_URL /content/repo
os.chdir('/content/repo')
!pip install -q -r requirements.txt
!nvidia-smi -L && ls

## 1. Dataset (Kaggle mirror of Kermany et al. 2018, CC BY 4.0)
Kaggle → Settings → API. Either **Generate New Token** (a string shown on screen — paste it below), or **Legacy API Credentials → Create New Token** (downloads `kaggle.json` — use the second cell). If the download returns 403, open the dataset page once in your browser and re-run.

In [ ]:
# Route 1: token string
import os
os.chdir('/content/repo')
os.environ['KAGGLE_API_TOKEN'] = 'paste-your-token-here'
!pip install -q -U kaggle
!python scripts/download_data.py kaggle
!cat data/raw/DOWNLOAD_INFO.json

In [ ]:
# Route 2: legacy kaggle.json file (skip if Route 1 worked)
import os
from google.colab import files
os.chdir('/content/repo')
os.makedirs('/root/.kaggle', exist_ok=True)
up = files.upload()                      # choose kaggle.json
open('/root/.kaggle/kaggle.json','wb').write(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!python scripts/download_data.py kaggle
!cat data/raw/DOWNLOAD_INFO.json

## 2. Audit + split + cache, five models, both ablations, report
Set `SEEDS="42 43 44"` for mean ± std (3× the time). If the session disconnects, just re-run this cell: finished runs are skipped.

In [ ]:
import os; os.chdir('/content/repo')
# main run (seed 42 is done and committed; teammates use 43 / 44). FRACTIONS="" skips the data-size ablation.
!SEEDS="43" EPOCHS=12 FRACTIONS="" bash scripts/run_all.sh
# extra check (optional, ~25 min): same models with squashed instead of padded images
# !RESIZE=stretch SEEDS="42" EPOCHS=12 FRACTIONS="" ABLATIONS="" bash scripts/run_all.sh

## 2b. Assignment 2 — paper-critique reality check (Singh et al., Sci. Rep. 2024)
Runs the paper's model (DeiT-B/16, ImageNet-pretrained, fully fine-tuned, LR 1e-5) under the A1 protocol, and the paper's protocol (random 80/10/10 split **by image**, no patient grouping) on the paper's model and on our A1 models. Needs cells 0 and 1 above (code + data). `STAGE=core` ≈ 50 min per seed on a T4; `STAGE=all` adds the LR check and the other models (≈ +40 min per seed). Finished runs are skipped, so re-run the cell after a disconnect.

In [ ]:
import os; os.chdir('/content/repo')
!pip install -q timm
!SEEDS="42 43" STAGE=core bash scripts/run_a2.sh
# then, if time allows:
# !SEEDS="42 43" STAGE=all bash scripts/run_a2.sh

### Assignment 2 results

In [ ]:
import os; os.chdir('/content/repo')
from IPython.display import Markdown, Image, display
display(Markdown(open('results/a2_reality_check.md').read()))
display(Image('results/figures/a2_split_effect.png'))

## 3. Look at the results

In [ ]:
import os; os.chdir('/content/repo')
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary.md').read()))
for f in ['results/figures/main_auroc.png','results/figures/ablation_permute.png','results/figures/ablation_fraction.png']:
    if os.path.exists(f): display(Image(f))

## 3. Project — weights written by an LLM, no training (see `astra/README.md`)
Needs cells B (clone) and the Kaggle cell. **3a** once; **3b** once per Astra practice pass (condition B only); **3c** once, after both Astra conditions are finished.

In [ ]:
# 3a. data cache + baselines (does NOT score Astra's weights): same architecture trained (seeds 42, 43) and random-init floor
import os; os.chdir('/content/repo')
!bash scripts/run_written.sh

In [ ]:
# 3b. ONE practice pass on practice images (never the test set). Upload Astra's file when asked.
import os, shutil; os.chdir('/content/repo'); os.makedirs('astra', exist_ok=True)
from google.colab import files
up = files.upload(); name = list(up)[0]
open(f'astra/{name}', 'wb').write(up[name])
!PYTHONPATH=src python -m cxr.written check --weights astra/$name
!PYTHONPATH=src python -m cxr.written practice --weights astra/$name --rotate --per_image --budget 20
# copy EVERYTHING printed above back to Astra

In [ ]:
# 3c. FINAL scoring, once, after conditions A and B are both finished. Upload weights_zeroshot.py and weights_practice_final.py first (cell 3b's upload, or the file browser).
import os; os.chdir('/content/repo')
!FINAL=1 bash scripts/run_written.sh
from IPython.display import Markdown, display; display(Markdown(open('results/written_table.md').read()))

In [ ]:
# 3d. Chance baselines (no training): 200 random networks, and best-of-10 random networks chosen on practice slices (Astra's budget), 100 repetitions
import os; os.chdir('/content/repo')
!PYTHONPATH=src python -m cxr.written floor --n 200
!PYTHONPATH=src python -m cxr.written search --reps 100 --passes 10
!PYTHONPATH=src python -m cxr.written report
from IPython.display import Markdown, display; display(Markdown(open('results/written_table.md').read()))

In [ ]:
# 3e. Condition C: export 16 labelled TRAINING images (8 + 8, matched shapes) for Astra, and download them
import os, shutil; os.chdir('/content/repo')
!PYTHONPATH=src python -m cxr.examples --n_per_class 8 --out astra/examples
shutil.make_archive('/content/astra_examples', 'zip', 'astra/examples')
from google.colab import files; files.download('/content/astra_examples.zip')

In [ ]:
# 3f. Condition C: check Astra's file, then score it ONCE on the official test folder (upload weights_examples.py when asked)
import os; os.chdir('/content/repo')
from google.colab import files
up = files.upload(); open('astra/weights_examples.py', 'wb').write(up[list(up)[0]])
!PYTHONPATH=src python -m cxr.written check --weights astra/weights_examples.py
!FINAL=1 bash scripts/run_written.sh
from IPython.display import Markdown, display; display(Markdown(open('results/written_table.md').read()))

In [ ]:
# 3g. Condition D: ONE practice pass (practice images only, separate log). Upload Astra's file when asked.
import os; os.chdir('/content/repo'); os.makedirs('astra/practice_D', exist_ok=True)
from google.colab import files
up = files.upload(); name = list(up)[0]
open(f'astra/practice_D/{name}', 'wb').write(up[name])
!PYTHONPATH=src python -m cxr.written check --weights astra/practice_D/$name
!PYTHONPATH=src python -m cxr.written practice --weights astra/practice_D/$name --log astra/practice_log_D.jsonl --rotate --per_image --budget 20
# copy EVERYTHING printed above back to Astra

## 3E. Condition E — Astra edits the final layer of our trained ResNet-50 (no gradients). See `astra/PROMPT_E_edit.md`.

In [ ]:
# E1. once: train ResNet-50 seed 42 if this session has no checkpoint (~5 min; results to results/retrain/), compute features, export the final layer, show PASS 0 (practice = patient-split half of the official test folder)
import os; os.chdir('/content/repo'); os.makedirs('astra/edit', exist_ok=True)
![ -f data/cache/images_224.npy ] || PYTHONPATH=src python -m cxr.prepare_data --source data/raw --out data --seed 42 --img_size 224
![ -f runs/resnet50__abl-none__frac-1__seed-42/best.pt ] || PYTHONPATH=src python -m cxr.train --model resnet50 --seed 42 --epochs 12 --img_size 224 --results results/retrain
!PYTHONPATH=src python -m cxr.edit features
!PYTHONPATH=src python -m cxr.edit export
from google.colab import files; files.download('astra/edit/resnet50_fc.txt')
!PYTHONPATH=src python -m cxr.edit practice > astra/edit/pass_00_output.txt; cat astra/edit/pass_00_output.txt
# attach resnet50_fc.txt to Astra and paste everything printed from 'PASS 0' to the end of the table under the prompt

In [ ]:
# E2. ONE practice pass. Upload Astra's edit file when asked; paste EVERYTHING printed back to Astra.
import os; os.chdir('/content/repo')
from google.colab import files
up = files.upload(); name = list(up)[0]
open(f'astra/edit/{name}', 'wb').write(up[name])
!PYTHONPATH=src python -m cxr.edit practice --edit astra/edit/$name

In [ ]:
# E3. FINAL, once: upload edit_final.py; scores unedited vs threshold rival vs Astra's edit on the scoring half of the official test folder
import os; os.chdir('/content/repo')
from google.colab import files
up = files.upload(); open('astra/edit/edit_final.py', 'wb').write(up[list(up)[0]])
!PYTHONPATH=src python -m cxr.edit evaluate --edit astra/edit/edit_final.py
from IPython.display import Markdown, display; display(Markdown(open('results/edit_resnet50.md').read()))

## F. Condition F: example images + practice passes with CHANNEL DIAGNOSTICS
Prompt: `astra/PROMPT_F_diagnostics.md`. `RUN` numbers independent Astra chats (1, 2, 3); each run has its own folder and log.

In [ ]:
# F1. once per session: data cache (skipped if present) and the run folder
RUN = 1
import os; os.chdir('/content/repo')
![ -f data/cache/images_224.npy ] || PYTHONPATH=src python -m cxr.prepare_data --source data/raw --out data --seed 42 --img_size 224
os.makedirs(f'astra/practice_F{RUN}', exist_ok=True); print('ready: astra/practice_F%d' % RUN)

In [ ]:
# F2. ONE practice pass. First save Astra's file in its own cell, starting with the line
#     %%writefile /content/repo/astra/practice_F1/weights_diag_v1.py      (change the run and version numbers)
# then set N to the same version number and run this cell. Paste EVERYTHING it prints back to Astra.
RUN, N = 1, 1
import os; os.chdir('/content/repo')
f = f'astra/practice_F{RUN}/weights_diag_v{N}.py'
!PYTHONPATH=src python -m cxr.written check --weights {f}
!PYTHONPATH=src python -m cxr.written practice --weights {f} --log astra/practice_log_F{RUN}.jsonl --rotate --per_image --diagnose --budget 20

In [ ]:
# F3. FINAL, once per run: save Astra's FINAL file as astra/practice_F1/weights_diag_final.py with %%writefile, then run.
# Scores it on the test folder, runs the matched random search (same number of practice passes), downloads a zip.
RUN = 1
import os; os.chdir('/content/repo')
!PYTHONPATH=src python -m cxr.written evaluate --weights astra/practice_F{RUN}/weights_diag_final.py --label astra-diagnose-r{RUN} --log astra/practice_log_F{RUN}.jsonl
n = sum(1 for _ in open(f'astra/practice_log_F{RUN}.jsonl'))
![ -f results/written__search{n}.json ] || PYTHONPATH=src python -m cxr.written search --reps 100 --passes {n}
!PYTHONPATH=src python -m cxr.written report && cat results/written_table.md
!zip -qr /content/condition_F{RUN}.zip astra/practice_F{RUN} astra/practice_log_F{RUN}.jsonl results/written__astra-diagnose-r{RUN}.json results/written__search{n}.json results/written_table.md
from google.colab import files; files.download(f'/content/condition_F{RUN}.zip')

## 5. External test: RSNA Pneumonia (adults, other hospitals)
One-time step on kaggle.com first: open https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules and click **I Understand and Accept** (same Kaggle account as your token). The download is about 3.9 GB. Needs the Kaggle cell above to have run in this session.

In [ ]:
# 5. download RSNA, train any missing checkpoints (results go to results/retrain/), score every model on 1,000 RSNA images
import os; os.chdir('/content/repo')
![ -f rsna-pneumonia-detection-challenge.zip ] || kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/repo
!bash scripts/run_external.sh
from IPython.display import Markdown, display; display(Markdown(open('results/external_rsna.md').read()))

## 4. Save the artefacts (results JSONs, tables, figures, audit) — send these back and commit them to the repo

In [ ]:
import os; os.chdir('/content/repo')
from google.colab import files
!zip -qr /content/a1_results.zip results astra data/audit.md data_ext/rsna/index.csv data/audit.json data/index.csv data/raw/DOWNLOAD_INFO.json
files.download('/content/a1_results.zip')

## R. Robustness check: size-matched test subset
Pairs each normal test X-ray with a pneumonia X-ray of nearly the same original width and height (within ~5 %), so image size and aspect ratio no longer separate the classes, then re-scores the LLM-written networks and the trained ones. Trains missing checkpoints first (~12 min on a T4).

In [ ]:
# R1. needs the data cache (cell F1 or the A1 prepare step); downloads results/robust_size.*
import os; os.chdir('/content/repo')
![ -f data/cache/images_224.npy ] || PYTHONPATH=src python -m cxr.prepare_data --source data/raw --out data --seed 42 --img_size 224
!PYTHONPATH=src python -m cxr.robust run
!zip -qr /content/robust.zip results/robust_size.json results/robust_size.md results/retrain
from google.colab import files; files.download('/content/robust.zip')

## G. Condition G: Astra writes the final layer of a FROZEN ImageNet ResNet-50
Prompt: `astra/PROMPT_G_resnet_head.md`. Same practice loop and report as F, one row per feature (2,048). No images attached.

In [ ]:
# G1. once per session: data cache, frozen ImageNet ResNet-50 features (~3 min on a T4), then PASS 0 (not counted)
import os; os.chdir('/content/repo'); os.makedirs('astra/head', exist_ok=True)
![ -f data/cache/images_224.npy ] || PYTHONPATH=src python -m cxr.prepare_data --source data/raw --out data --seed 42 --img_size 224
![ -f astra/head/features.npz ] || PYTHONPATH=src python -m cxr.head features
!PYTHONPATH=src python -m cxr.head practice > astra/head/pass_00_output.txt; head -4 astra/head/pass_00_output.txt; sed -n '/FEATURE REPORT/,+8p' astra/head/pass_00_output.txt
from google.colab import files; files.download('astra/head/pass_00_output.txt')   # paste or attach this to Astra

In [ ]:
# G2. ONE practice pass. First save Astra's file in its own cell, starting with the line
#     %%writefile /content/repo/astra/head/head_v1.py        (change the version number)
# then set N to the same number and run. The output is long: it is saved and downloaded as a .txt to attach.
N = 1
import os; os.chdir('/content/repo')
f = f'astra/head/head_v{N}.py'
out = f'astra/head/pass_{N:02d}_output.txt'
!PYTHONPATH=src python -m cxr.head practice --head {f} > {out}; head -3 {out}
from google.colab import files; files.download(out)

In [ ]:
# G3. FINAL, once: save Astra's FINAL file as astra/head/head_final.py with %%writefile, then run.
# Scores it on the test folder next to the rivals (matched random search, selection rule, trained linear probe), downloads a zip.
import os; os.chdir('/content/repo')
!PYTHONPATH=src python -m cxr.head evaluate --head astra/head/head_final.py
!zip -qr /content/condition_G.zip astra/head/*.py astra/head/*.txt astra/practice_log_G1.jsonl results/head_resnet50.*
from google.colab import files; files.download('/content/condition_G.zip')

## H. Condition H: Astra EDITS our trained ResNet-50's last layer, with the per-feature report
Prompt: `astra/PROMPT_H_edit_report.md`. Practice = one patient-split half of the test folder (306 images); scoring = the other half (312).

In [ ]:
# H1. once per session (~10 min if the ResNet-50 must be trained): data cache, checkpoint, features, weight file, PASS 0
import os; os.chdir('/content/repo'); os.makedirs('astra/edit', exist_ok=True); os.makedirs('astra/edit_h', exist_ok=True)
![ -f data/cache/images_224.npy ] || PYTHONPATH=src python -m cxr.prepare_data --source data/raw --out data --seed 42 --img_size 224
![ -f runs/resnet50__abl-none__frac-1__seed-42/best.pt ] || PYTHONPATH=src python -m cxr.train --model resnet50 --seed 42 --epochs 12 --img_size 224 --results results/retrain
![ -f astra/edit/features.npz ] || PYTHONPATH=src python -m cxr.edit features
!PYTHONPATH=src python -m cxr.edit export
!PYTHONPATH=src python -m cxr.edit practice --report > astra/edit_h/pass_00_output.txt; head -4 astra/edit_h/pass_00_output.txt
from google.colab import files; files.download('astra/edit/resnet50_fc.txt'); files.download('astra/edit_h/pass_00_output.txt')

In [ ]:
# H2. ONE practice pass. First save Astra's file in its own cell, starting with the line
#     %%writefile /content/repo/astra/edit_h/edit_v1.py        (change the version number)
# then set N to the same number and run. Attach the downloaded .txt to Astra.
N = 1
import os; os.chdir('/content/repo')
f = f'astra/edit_h/edit_v{N}.py'
out = f'astra/edit_h/pass_{N:02d}_output.txt'
!PYTHONPATH=src python -m cxr.edit practice --report --edit {f} --log astra/practice_log_H.jsonl > {out}; head -3 {out}
from google.colab import files; files.download(out)

In [ ]:
# H3. FINAL, once: save Astra's FINAL file as astra/edit_h/edit_final.py with %%writefile, then run.
import os; os.chdir('/content/repo')
!PYTHONPATH=src python -m cxr.edit evaluate --edit astra/edit_h/edit_final.py --log astra/practice_log_H.jsonl --probe --tag H
!zip -qr /content/condition_H.zip astra/edit_h astra/practice_log_H.jsonl results/edit_resnet50_H.*
from google.colab import files; files.download('/content/condition_H.zip')